In [10]:
import os
from pyexpat.errors import messages
from typing import List

from openai import OpenAI
from  dotenv import load_dotenv

_ = load_dotenv()

In [4]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [5]:
def get_completions(prompt, model):
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0
    )

    return response


In [6]:

get_completions("What is 1+1?", "gpt-4o-mini")

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [20]:
import requests
import time
import os
from  dotenv import load_dotenv

_ = load_dotenv()

start = time.perf_counter()
llm_url = os.getenv("LLM_URL")
url = llm_url + "api/chat"
payload = {
    "model": "mistral",
    "messages": [
        {"role": "user", "content": "Waht is 1+1?"}
    ],
    "stream": False
}

response = requests.post(url, json=payload) #, timeout=120)
response.raise_for_status()
print(response.json()["message"]["content"])
# print("text:", response.text)
end = time.perf_counter()
print(end - start)

 The sum of 1+1 is 2.
3.566469811999923


In [21]:
from openai import OpenAI

client = OpenAI(
    base_url=os.getenv("LLM_URL") + r"v1/",
    api_key="ollama"
)

# openai sdk compatible ollama call
def get_completion(prompt, model):
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    response = client.chat.completions.create(
        model= model,
        messages=messages,
        temperature=0
    )

    # return response.choices[0].message["content"]
    return response.choices[0].message.content



In [22]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse,\
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [23]:
style = """
American English \
in a calm and respectful tone
"""

In [24]:
prompt = f"""
Translate the text \
that is delimited by the triple backticks \
into a style that is {style}.
text: ```{customer_email}```
"""
print(prompt)


Translate the text that is delimited by the triple backticks into a style that is 
American English in a calm and respectful tone
.
text: ```
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse,the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!
```



In [25]:
output = get_completion(prompt, "mistral")
print(output)

 "Ah, I'm absolutely livid! My blender lid flew off and splattered my kitchen walls with smoothie! To add insult to injury, the warranty doesn't cover the cost of cleaning up my kitchen. I really need your assistance right now, friend."


# Using Langchain

In [26]:
# !pip install langchain-ollama

In [27]:
from langchain_ollama import ChatOllama

In [28]:
chat = ChatOllama(
    model='mistral',
    temperature=0,
    base_url=os.getenv("LLM_URL"),
)

## Prompt Template




In [29]:
template_string = """Translate the text \
that is delimited by the triple backticks \
into a style that is {style}.
text: ```{text}```
"""

In [30]:
# !pip install langchain

In [31]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(template_string)

In [32]:
prompt_template.messages[0].prompt

PromptTemplate(input_variables=['style', 'text'], input_types={}, partial_variables={}, template='Translate the text that is delimited by the triple backticks into a style that is {style}.\ntext: ```{text}```\n')

In [33]:
prompt_template.messages[0].prompt.input_variables

['style', 'text']

In [34]:
customer_style = """American English \
in a calm and respectful tone
"""

In [35]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [36]:
customer_messages = prompt_template.format_messages(
                    style=customer_style,
                    text=customer_email)

In [37]:
print(type(customer_messages))
print(type(customer_messages[0]))

<class 'list'>
<class 'langchain_core.messages.human.HumanMessage'>


In [38]:
print(customer_messages[0])

content="Translate the text that is delimited by the triple backticks into a style that is American English in a calm and respectful tone\n.\ntext: ```\nArrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!\n```\n" additional_kwargs={} response_metadata={}


In [39]:
customer_response = chat.invoke(customer_messages)

In [40]:
print(customer_response.content)

 "Ah, I'm absolutely furious! My blender lid flew off and splattered my kitchen walls with smoothie! To add insult to injury, the warranty doesn't cover the cost of cleaning up my kitchen. I really need your assistance right now, friend!"


In [41]:
service_reply = """Hey there customer, \
the warranty does not cover \
cleaning expenses for your kitchen \
because it's your fault that \
you misused your blender \
by forgetting to put the lid on before \
starting the blender. \
Tough luck! See ya!
"""

In [42]:
service_style_pirate = """\
a polite tone \
that speaks in English Pirate\
"""

In [43]:
service_messages = prompt_template.format_messages(
    style=service_style_pirate,
    text=service_reply)

print(service_messages[0].content)

Translate the text that is delimited by the triple backticks into a style that is a polite tone that speaks in English Pirate.
text: ```Hey there customer, the warranty does not cover cleaning expenses for your kitchen because it's your fault that you misused your blender by forgetting to put the lid on before starting the blender. Tough luck! See ya!
```



In [44]:
service_response = chat.invoke(service_messages)
print(service_response.content)

 Arrr matey! Me hearty greetings to ye! Now, 'tis a shame I must be the bearer of bad news, but I'm afraid yer warranty canna cover yer cleanin' expenses for yer kitchen, seein' as ye misused yer fine blender. Ye forgot to secure the lid afore startin' the blender, and that's a fact as certain as the sun risin' in the east! Aye, it's a tough break, but that's the way the wind blows, I'm afraid. Farewell for now, and may yer next adventure be a smoother one!


## Output Parsers

In [45]:
{
    "gift": False,
    "delivery_days": 5,
    "price_value": "pretty affordable"
}

{'gift': False, 'delivery_days': 5, 'price_value': 'pretty affordable'}

In [46]:
customer_review = """\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

In [49]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(review_template)

print(prompt_template)

input_variables=['text'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='For the following text, extract the following information:\n\ngift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.\n\ndelivery_days: How many days did it take for the product to arrive? If this information is not found, output -1.\n\nprice_value: Extract any sentences about the value or price,and output them as a comma separated Python list.\n\nFormat the output as JSON with the following keys:\ngift\ndelivery_days\nprice_value\n\ntext: {text}\n'), additional_kwargs={})]


In [51]:
messages = prompt_template.format_messages(text=customer_review)
chat = ChatOllama(
    model='mistral',
    temperature=0.0,
    base_url=os.getenv("LLM_URL"),
)
response = chat.invoke(messages)

print(response.content)

 {
  "gift": "True",
  "delivery_days": "2",
  "price_value": ["It's slightly more expensive than the other leaf blowers out there", "It's worth it for the extra features."]
}


In [52]:
type(response.content)

str

In [53]:
response.content.get('gift')

AttributeError: 'str' object has no attribute 'get'

### Parse the LLM output string into a Python dictionary using pydantic

In [61]:
# from langchain.output_parsers import ResponseSchema
# from langchain.output_parsers import StructuredOutputParser
# Modern Alternative

In [66]:
from pydantic import BaseModel, Field
from typing import List

class ReviewData(BaseModel):
    gift: str = Field(description="True or False")
    delivery_days: str = Field(description="How many days did it take for the product")
    price_value: List[str] = Field(description="Extract any sentences about the value or price")

structured_response = chat.with_structured_output(ReviewData)

result = structured_response.invoke(messages)

In [67]:
print(result.gift)

True


In [68]:
print(result.delivery_days)

2


In [69]:
print(result.price_value)

["It's slightly more expensive than the other leaf blowers out there", "It's worth it for the extra features."]
